# Comparing classifiers for bank telemarketing
**Venky Ravipati · Practical Application 17.1**

A campaign manager has enough agents to call only a fraction of the eligible list. Which pre-call records should receive priority if the goal is to find term-deposit subscribers? Compare four classifiers against untargeted calling. Historical subscription predicts response among previously contacted people. Measuring the additional effect of a call requires a controlled campaign.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
from time import perf_counter

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, average_precision_score
from statsmodels.stats.proportion import confint_proportions_2indep

sns.set_theme(style="whitegrid")
seed = 17

## Data and cleaning
Use UCI’s full, date-ordered bank-additional file (May 2008 through November 2010). Keep explicit unknown categories. Identical rows may represent separate calls because the file has no customer identifier. Preserve them and report their count. Reserve the last fifth of the records for a later-period evaluation before exploring outcomes.

In [ ]:
filename = "bank-additional-full.csv"
local_files = list(Path(".").rglob(filename))
if local_files:
    df = pd.read_csv(local_files[0], sep=";")
else:
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    with urlopen(url, timeout=60) as response, ZipFile(BytesIO(response.read())) as outer:
        inner_name = next(n for n in outer.namelist() if n.endswith("bank-additional.zip"))
        with ZipFile(BytesIO(outer.read(inner_name))) as inner:
            member = next(n for n in inner.namelist() if n.endswith("/" + filename))
            df = pd.read_csv(inner.open(member), sep=";")
assert df.shape == (41188, 21) and set(df.y.unique()) == {"yes", "no"}
assert df.isna().sum().sum() == 0
cut = int(len(df) * .8)
train = df.iloc[:cut].copy()
print(f"{len(df):,} records | {df.duplicated().sum()} exact repeats | {cut:,} training records")
display(train.eq("unknown").sum().loc[lambda s: s.gt(0)].rename("Unknowns in training").to_frame())

## Campaign patterns in the training period

In [ ]:
rates = train.groupby("poutcome").y.agg(calls="size", subscribers=lambda s: s.eq("yes").sum())
rates["subscription_rate"] = rates.subscribers / rates.calls
display(rates.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=rates.index, y=rates.subscription_rate * 100, ax=axes[0], color="#247778")
axes[0].set(xlabel="Previous campaign outcome", ylabel="Subscriptions (%)", title="Response by prior outcome")
sns.histplot(data=train, x="age", hue="y", bins=28, stat="density", common_norm=False,
             element="step", fill=False, ax=axes[1])
axes[1].set(xlabel="Client age (years)", ylabel="Within-group density", title="Age distribution by subscription")
plt.tight_layout()
plt.show()
success, failure = rates.loc["success"], rates.loc["failure"]
gap = success.subscription_rate - failure.subscription_rate
lo, hi = confint_proportions_2indep(int(success.subscribers), int(success.calls),
                                    int(failure.subscribers), int(failure.calls),
                                    method="newcomb", compare="diff")
display(Markdown(f"Prior success versus failure was associated with a **{gap*100:.1f} percentage-point** difference in subscription rate (approximate 95% interval {lo*100:.1f} to {hi*100:.1f} points). This unadjusted comparison assumes independent records. Repeat customers and campaign timing may affect both the association and its uncertainty."))

## Features available at call planning
Call duration is learned after contact. Exclude it. The recorded campaign count includes the current contact, so use the number already attempted. Replace the 999-day sentinel with a separate indicator and zero-fill elapsed days. Assume planned channel, weekday, month and the recorded macroeconomic indicators are available when scoring. Verify publication lags before deployment.

In [ ]:
X = df.drop(columns=["duration", "y"]).copy()
X["previous_attempts_this_campaign"] = X.pop("campaign") - 1
X["has_prior_contact_date"] = X.pdays.ne(999).astype(int)
X["pdays"] = X.pdays.replace(999, 0)
y = df.y.eq("yes").astype(int)
Xtr, Xte = X.iloc[:cut], X.iloc[cut:]
ytr, yte = y.iloc[:cut], y.iloc[cut:]
categorical = Xtr.select_dtypes(exclude="number").columns
numeric = Xtr.select_dtypes(include="number").columns
prep = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])

## Validation, tuning and baseline
Use three expanding time splits inside the training period. Fit encoders and scalers within each fold. Select by average precision (AP), which rewards placing subscribers early in the call list across recall levels. A constant score has AP equal to the positive rate. Also compare training and test accuracy, fitting time, and the yield of calling the highest-ranked 10%. The capacity limit is an illustrative business assumption.

In [ ]:
models = {
    "Logistic regression": (LogisticRegression(max_iter=1500), {"C": [0.1, 1]}),
    "K-nearest neighbors": (KNeighborsClassifier(n_jobs=1), {"n_neighbors": [15, 45]}),
    "Decision tree": (DecisionTreeClassifier(min_samples_leaf=40, random_state=seed),
                      {"max_depth": [3, 6, 10]}),
    "Linear support vector machine": (LinearSVC(dual=False, max_iter=5000, random_state=seed),
                                      {"C": [0.01, 0.1, 1]}),
}
fits, rows = {}, []
for name, (model, grid) in models.items():
    pipe = Pipeline([("prep", prep), ("model", model)])
    search = GridSearchCV(pipe, {"model__" + k: v for k, v in grid.items()},
                          scoring="average_precision", cv=TimeSeriesSplit(3), error_score="raise")
    start = perf_counter()
    search.fit(Xtr, ytr)
    elapsed = perf_counter() - start
    fits[name] = search.best_estimator_
    rows.append([name, search.best_params_, search.best_score_, elapsed])
validation = pd.DataFrame(rows, columns=["Model", "Parameters", "CV AP", "Search seconds"]).set_index("Model")
winner = validation["CV AP"].idxmax()
display(validation.sort_values("CV AP", ascending=False).round(3))

## Later-period results
Fix the model choice before opening the holdout. Randomize ties reproducibly so that a tree leaf or tied neighbor vote does not receive priority simply because its records appear earlier in the file.

In [ ]:
n_calls = int(np.ceil(.1 * len(yte)))
tie_order = np.random.default_rng(seed).permutation(len(yte))
rows = []
for name, model in fits.items():
    scores = (model.decision_function(Xte) if hasattr(model, "decision_function")
              else model.predict_proba(Xte)[:, 1])
    top = np.lexsort((tie_order, -scores))[:n_calls]
    hits = yte.iloc[top].sum()
    rows.append([name, accuracy_score(ytr, model.predict(Xtr)), accuracy_score(yte, model.predict(Xte)),
                 average_precision_score(yte, scores), hits / n_calls, hits / yte.sum()])
results = pd.DataFrame(rows, columns=["Model", "Train accuracy", "Test accuracy", "Test AP",
                                      "Top 10% yield", "Top 10% recall"]).set_index("Model")
results.loc["Majority / random priority"] = [1-ytr.mean(), 1-yte.mean(), yte.mean(), yte.mean(), n_calls/len(yte)]
display(results.round(3))
ax = results["Test AP"].sort_values().plot.barh(color="#247778", figsize=(8, 3.5))
ax.set(xlabel="Average precision", ylabel="", title="Ranking performance on later campaigns", xlim=(0, 1))
plt.tight_layout()
plt.show()

## Interpreting the logistic model

In [ ]:
logit = fits["Logistic regression"]
names = logit.named_steps["prep"].get_feature_names_out()
coefficients = pd.Series(logit.named_steps["model"].coef_[0], index=names)
age_effect = np.exp(coefficients["num__age"])
age_sd = logit.named_steps["prep"].named_transformers_["num"].scale_[list(numeric).index("age")]
display(Markdown(f"Holding the other inputs fixed, a {age_sd:.1f}-year increase in age multiplies the fitted odds of subscription by **{age_effect:.3f}**. This summarizes a regularized association. It does not establish that age causes a change in response."))

## Findings, recommendations and next steps

In [ ]:
selected = results.loc[winner]
ratio = selected["Top 10% yield"] / yte.mean()
summary = (
    f"{winner} led expanding-window validation (mean AP {validation.loc[winner, 'CV AP']:.3f}). "
    f"On {len(yte):,} later records its AP was {selected['Test AP']:.3f}, against a constant-score "
    f"baseline of {yte.mean():.3f}. Calling the first {n_calls:,} ranked records would have reached "
    f"{selected['Top 10% yield']:.1%} subscribers per call versus {yte.mean():.1%} under random "
    f"priority, or {ratio:.2f} times the expected yield. This list captured "
    f"{selected['Top 10% recall']:.1%} of observed subscribers."
)
action = ("Pilot the ranked list against random priority at equal call capacity." if ratio > 1 else
          "Keep random priority while investigating the drop in ranking performance.")
next_steps = (action + " Track subscriptions per 100 calls, agent time, opt-outs and net margin. "
              "Validate across new months and customer groups. Use a randomized no-call group to "
              "measure incremental subscriptions before expanding the campaign.")
display(Markdown(summary + "\n\n" + next_steps))
readme = ("# Bank telemarketing | Comparing classifiers\n\n"
          "[Open the analysis notebook](comparing_classifiers.ipynb)\n\n"
          "## Campaign findings\n\n" + summary + "\n\n" + next_steps +
          "\n\n## Reproduce\n\nInstall requirements.txt, open the notebook from the repository root "
          "and run all cells. It reads bank-additional-full.csv locally or downloads the UCI archive. "
          "The last cell updates this summary.\n\n"
          "Source: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), "
          "Moro, Cortez and Rita.\n")
Path("README.md").write_text(readme, encoding="utf-8");